# Pandas Basics — Part 3

In this lesson, we're going to introduce more fundamentals of [Pandas](https://pandas.pydata.org/pandas-docs/stable/getting_started/overview.html), a powerful Python library for working with tabular data like CSV files.

We will review skills learned from the last two lessons and introduce how to:

- Check for duplicate data
- Clean and transform data
- Manipulate string data
- Apply functions
- Reset index
- Bonus! Create an interactive data viz

___

## Dataset
### *The Pudding*'s Film Dialogue Data

The dataset that we're working with in this lesson is taken from Hannah Andersen and Matt Daniels's *Pudding* essay, ["Film Dialogue from 2,000 screenplays, Broken Down by Gender and Age"](https://pudding.cool/2017/03/film-dialogue/). The dataset provides information about 2,000 films from 1925 to 2015, including characters’ names, genders, ages, how many words each character spoke in each film, the release year of each film, and how much money the film grossed. They included character gender information because they wanted to contribute data to a broader conversation about how "white men dominate movie roles."

Yet transforming complex social constructs like gender into quantifiable data is tricky and historically fraught. They claim, in fact, that one of the [most frequently asked questions](https://medium.com/@matthew_daniels/faq-for-the-film-dialogue-by-gender-project-40078209f751) about the piece is about gender: “Wait, but let’s talk about gender. How do you know the monster in Monsters Inc. is a boy!" The short answer is that they don't. To determine character gender, they used actors' IMDB information, which they acknowledge is an imperfect approach: "Sometimes, women voice male characters. Bart Simpson, for example, is voiced by a woman. We’re aware that this means some of the data is wrong, AND we’re still fine with the methodology and approach."

As we work with this data, we want to be critical and cognizant of this approach to gender. How does such a binary understanding of gender, gleaned from the IMDB pages of actors, influence our later results and conclusions? What do we gain by using such an approach, and what do we lose? How else might we have encoded or determined gender for the same data? 

___

**Import Pandas**

To use the Pandas library, we first need to `import` it.

In [ ]:
import pandas as pd

The above `import` statement not only imports the Pandas library but also gives it an alias or nickname — `pd`. This alias will save us from having to type out the entire words `pandas` each time we need to use it. Many Python libraries have commonly used aliases like `pd`.

**Set Display Settings**

By default, Pandas will display 60 rows and 20 columns. I often change [Pandas' default display settings](https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html) to show more rows or columns.

In [ ]:
pd.options.display.max_rows = 100

**Read in CSV File**

To read in a CSV file, we will use the function `pd.read_csv()` and insert the name of our desired file path. 

In [ ]:
film_df = pd.read_csv('./data/Pudding-Film-Dialogue-Salty.csv', delimiter=",", encoding='utf-8')

When reading in the CSV file, we also specified the `encoding` and `delimiter`. The `delimiter` parameter specifies the character that separates or "delimits" the columns in our dataset. For CSV files, the delimiter will most often be a comma. (CSV is short for *Comma Separated Values*.) Sometimes, however, the delimiter of a CSV file might be a tab (`\t`) or, more rarely, another character.

**Display Data**

We can display a DataFrame in a Jupyter notebook simply by running a cell with the variable name of the DataFrame.

In [ ]:
film_df

In [ ]:
film_df.describe(include='all')

Do you notice any outliers, anomalies, or potential problems here?

The maximum value in the "age" column is 2013! That seems like an error.

In [ ]:
film_df[film_df['age'] == 2013]

Let's drop this row from the dataset by using the `.drop()` method and the Index number of the row.

In [ ]:
film_df = film_df.drop(11639)

Now if we look for it again, that row is gone.

In [ ]:
film_df[film_df['age'] == 2013]

## Clean and Transform Data

### Pandas `.str` Methods

Remember all the special things that you can do with Python strings aka [string methods](https://melaniewalsh.github.io/Intro-Cultural-Analytics/Python/String-Methods.html)?

Pandas has special [Pandas string methods](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html#string-methods), too. Many of them are very similar to Python string methods, except they will transform every single string value in a column, and we have to add `.str` to the method chain.

| **Pandas String Method** | **Explanation**                                                                                   |
|:-------------:|:---------------------------------------------------------------------------------------------------:|
| df['column_name']`.str.lower()`         | makes the string in each row lowercase                                                                                |
| df['column_name']`.str.upper()`         | makes the string in each row uppercase                                                |
| df['column_name']`.str.title()`         | makes the string in each row titlecase                                                |
| df['column_name']`.str.replace('old string', 'new string')`      | replaces `old string` with `new string` for each row |
| df['column_name']`.str.contains('some string')`      | tests whether string in each row contains "some string" |
| df['column_name']`.str.split('delim')`          | returns a list of substrings separated by the given delimiter |
| df['column_name']`.str.join(list)`         | opposite of split(), joins the elements in the given list together using the string                                                                        |
                                                            

For example, to transform every character's name in the "character" column from lowercase to uppercase, we can use `.str.upper()` 

In [ ]:
film_df['imdb_character_name'].str.upper()

To transform every character's name in the "character" column to lowercase, we can use `.str.lower()`

In [ ]:
film_df['imdb_character_name'].str.lower()

If we want to replace the gender columns's single letter abbreviation for "male" / "female" (sex) with "man" / "woman" (gender identity), we could use the `.str.replace()` method. 

In [ ]:
film_df['gender'] = film_df['gender'].str.replace('m', 'man')

In [ ]:
film_df['gender'] = film_df['gender'].str.replace('f', 'woman')

In [ ]:
film_df.sample(10)

We can use the `.str.contains()` to search for particular words or phrases in a column, such as "Star Wars."

In [ ]:
film_df[film_df['title'].str.contains('Star Wars')]

We can use the `.str.contains()` to search for particular words or phrases in a column, such as "Mean Girls."

In [ ]:
film_df[film_df['title'].str.contains('Mean Girls')]

**Filter DataFrame**

We can filter the DataFrames for only characters who are men or women.

In [ ]:
men_film_df = film_df[film_df['gender'] == 'man']

In [ ]:
women_film_df = film_df[film_df['gender'] == 'woman']

**Groupby**

We can use the `.groupby()` function to group all the men characters in each film and sum up their total dialogue.

By adding a Python string slice, we can identify the top 20 films with the greatest proportion of men speaking.

Line Breaks: If a line of code gets too long, you can create a line break with a backslash `\`

In [ ]:
men_film_df.groupby('title')[['proportion_of_dialogue']]\
.sum().sort_values(by='proportion_of_dialogue', ascending=False)[:20]

From the *Pudding* paper:

"For each screenplay, we mapped characters with at least 100 words of dialogue to a person’s IMDB page (which identifies people as an actor or actress). We did this because minor characters are poorly labeled on IMDB pages. This has unintended consequences: Schindler’s List, for example, has women with lines, just not over this threshold. Which means a more accurate result would be 99.5% male dialogue instead of our result of 100%. There are other problems with this approach as well: films change quite a bit from script to screen. Directors cut lines. They cut characters. They add characters. They change character names. They cast a different gender for a character. We believe the results are still directionally accurate, but individual films will definitely have errors."

We can use the `.groupby()` function to group all the women characters in each film and sum up their total dialogue.

By adding a Python string slice, we can identify the top 20 films with the greatest proportion of women speaking.

In [ ]:
women_film_df.groupby('title')[['proportion_of_dialogue']]\
.sum().sort_values(by='proportion_of_dialogue', ascending=False)[:20]

## Reset Index

We can transform a Groupby object into a DataFrame with a regular Index by tacking on `.reset_index()`.

In [ ]:
women_film_df.groupby('title')[['proportion_of_dialogue']]\
.sum().sort_values(by='proportion_of_dialogue', ascending=False).reset_index()

## Bonus — Interactive Data Visualization

In [ ]:
#Import necessary Bokeh modules
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, NumeralTickFormatter
from bokeh.io import output_notebook, show
from bokeh.palettes import RdBu
from bokeh.transform import linear_cmap, jitter

In [ ]:
#Set up Bokeh to work in Jupyter notebook
output_notebook()

Here's an interactive data visualization of these Hollywood films by release year and percentage of women dialogue. This data viz was created with the Python library [Bokeh](https://docs.bokeh.org/en/latest/index.html), which we will discuss in some later lessons.

* Scroll to zoom in
* Hover to see more information about each point

To see the code that created this visualization, select "Click to show" below.

In [ ]:
#Make groupby into a new DataFrame
dialogue_df = women_film_df.groupby(['title', 'release_year'])[['proportion_of_dialogue']].sum()\
.sort_values(by='proportion_of_dialogue', ascending=False).reset_index()

# Set up the source data that will suppply the x,y columns and the film title hover text
source = ColumnDataSource(dialogue_df)

# Set the hover tool tip to the film title, release year, and proportion of dialogue

TOOLTIPS = [("Title", "@title"),
            ("Year", "@release_year"),
           ("Women Dialogue", "@{proportion_of_dialogue}{%0.2f}")]

#Set up Bokeh plot with title, labels, 
bokeh_plot = figure(title="How Much Do Women Speak in Hollywood Films?", x_axis_label = 'Film Release Year',
                    y_axis_label = 'Amount of Dialogue Spoken By Women',x_range = [1930, 2018], y_range = [0, 1.01],
                 tooltips=TOOLTIPS, width=800, height=550, active_scroll='wheel_zoom')

# Create a red to blue color palette
color_mapper = linear_cmap(field_name='proportion_of_dialogue', palette=RdBu[4], low=1.1, high=0)

# Supply inidivudal points values
bokeh_plot.circle(y='proportion_of_dialogue', x=jitter('release_year', width=.2),
         size = 10,
        line_color='black',
        line_alpha=.4,
         source=source,
         color=color_mapper, alpha=.5)

bokeh_plot.title.text_font_size='20pt'

#Make Y axis percentages
bokeh_plot.yaxis.formatter = NumeralTickFormatter(format='0 %')

show(bokeh_plot)